In [ ]:
#Snail + TrackID + Static Data Display
from ultralytics import YOLO
from pathlib import Path
import cv2
import torch
import gc
import numpy as np
from ultralytics.utils.plotting import Annotator, colors
from collections import deque
import math

# --- Helper function to calculate distance ---
def calculate_distance(p1, p2):
    """Calculate Euclidean distance between two (x, y) points."""
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

INPUT_DIR = r"C:\Users\gifly\Desktop\Playground\TGR2025\P3_VIDEO__DAY3.mp4"
MODEL_PATH = r"C:\Users\gifly\Desktop\Playground\TGR2025\longest.pt"

input_dir = Path(INPUT_DIR)
model = YOLO(MODEL_PATH)

# Open the video file
cap = cv2.VideoCapture(str(input_dir))
if not cap.isOpened():
    raise IOError(f"Cannot open video file: {input_dir}")

# Get video properties
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
# (Optional) Video writer
# output_path = "output_fast_tracker.mp4"
# fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# --- Custom Tracker State ---
tracks = {
    1: {'pos': None, 'history': [], 'age': 0, 'box': None, 'first_seen_frame': -1},
    2: {'pos': None, 'history': [], 'age': 0, 'box': None, 'first_seen_frame': -1}
}

# --- Tunable Parameters ---
MAX_AGE = 50 
DISTANCE_THRESHOLD = 30 
FRAME_SKIP = 3  # <-- Number of frames to skip. 3 is a good start.
                # Increase for more speed, decrease for more accuracy.

# --- NEW: Check for GPU and set half precision ---
USE_HALF_PRECISION = torch.cuda.is_available()
print(f"Is CUDA (GPU) available? {torch.cuda.is_available()}")
if not USE_HALF_PRECISION:
    print("!!! WARNING: Running on CPU. This will be very slow. !!!")


# <--- NEW: Static Data for Top-Left Display ---
# Put your actual lat/lon/alt data here, one entry for each ID
static_track_data = {
    1: {"lat": 14.30485, "lon": 101.17280, "alt": 40.52},
    2: {"lat": 14.30485, "lon": 101.17280, "alt": 40.52} 
    # ^ You can update these values from another source if they change
}

# <--- NEW: Font, color, and position settings for the text ---
font = cv2.FONT_HERSHEY_SIMPLEX
font_scale = 0.6
line_thickness = 2
start_x = 20  # Pixels from the left edge
start_y = 40  # Pixels from the top edge
line_height = 25 # Vertical pixels between each line

frame_count = 0 # Keep track of current frame number

# Process the video frame by frame
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break
    
    frame_count += 1
    
    # --- LOGIC CHANGE 1: Increment age for ALL tracks first ---
    # Every track gets 1 frame older
    for track_id in [1, 2]:
        tracks[track_id]['age'] += 1

    # --- LOGIC CHANGE 2: Only run prediction/matching on specified frames ---
    if frame_count % FRAME_SKIP == 0:
        # --- Run Prediction & Matching ---
        
        # <--- MODIFIED: Added imgsz=640 and half=... for speed ---
        results = model.predict(frame, imgsz=640, half=USE_HALF_PRECISION, verbose=False)

        current_detections = []
        if results and results[0].boxes:
            for box in results[0].boxes.xywh.cpu():
                x, y, w, h = box
                centroid = (int(x), int(y))
                bbox_xyxy = [x-w/2, y-h/2, x+w/2, y+h/2]
                current_detections.append({'centroid': centroid, 'box': bbox_xyxy})
        
        # --- Match Detections to Tracks ---
        unassigned_detections = list(range(len(current_detections)))
        tracks_updated = {1: False, 2: False}

        # Part A: Try to match existing, "live" tracks
        for track_id in [1, 2]:
            if tracks[track_id]['pos'] is not None and tracks[track_id]['age'] <= MAX_AGE:
                last_pos = tracks[track_id]['pos']
                best_dist = float('inf')
                best_det_idx = -1
                
                for i in unassigned_detections:
                    dist = calculate_distance(last_pos, current_detections[i]['centroid'])
                    if dist < DISTANCE_THRESHOLD and dist < best_dist:
                        best_dist = dist
                        best_det_idx = i
                
                if best_det_idx != -1:
                    det = current_detections[best_det_idx]
                    tracks[track_id]['pos'] = det['centroid']
                    # Don't append history here, we do it in a separate step
                    tracks[track_id]['age'] = 0 # Reset age because we found it
                    tracks[track_id]['box'] = det['box']
                    tracks_updated[track_id] = True
                    unassigned_detections.remove(best_det_idx)

        # Part B: Assign remaining detections to "empty" or "dead" tracks
        for det_idx in unassigned_detections:
            for track_id in [1, 2]:
                if not tracks_updated[track_id] and (tracks[track_id]['pos'] is None or tracks[track_id]['age'] > MAX_AGE):
                    det = current_detections[det_idx]
                    tracks[track_id]['pos'] = det['centroid']
                    tracks[track_id]['age'] = 0 # Reset age
                    tracks[track_id]['box'] = det['box']
                    tracks[track_id]['first_seen_frame'] = frame_count
                    tracks_updated[track_id] = True
                    break
    
    # --- END of `if frame_count % FRAME_SKIP == 0` block ---
    # The code below this line runs on EVERY frame

    # --- LOGIC CHANGE 3: Update history for all active tracks ---
    for track_id in [1, 2]:
        if tracks[track_id]['pos'] is not None and tracks[track_id]['age'] <= MAX_AGE:
            tracks[track_id]['history'].append(tracks[track_id]['pos'])


    # --- Draw Everything (This runs every frame) ---
    final_frame = frame.copy()
    
    # Draw trails (solid, full line)
    for track_id in [1, 2]:
        if tracks[track_id]['age'] < MAX_AGE and len(tracks[track_id]['history']) > 1:
            points = np.array(tracks[track_id]['history'], dtype=np.int32).reshape((-1, 1, 2))
            cv2.polylines(final_frame, [points], 
                          isClosed=False, 
                          color=colors(track_id, True), 
                          thickness=2)

    # Draw boxes and labels on top
    annotator = Annotator(final_frame, line_width=2)
    for track_id in [1, 2]:
        if tracks[track_id]['age'] < MAX_AGE and tracks[track_id]['box'] is not None:
            label = f"ID: {track_id}"
            color = colors(track_id, True)
            annotator.box_label(tracks[track_id]['box'], label, color=color)

    final_frame = annotator.result()

    # <--- NEW: Draw the Top-Left Static Text ---
    current_y = start_y # Reset the Y position for each frame
    
    for track_id in [1, 2]:
        # Only display info if the track is currently "alive"
        if tracks[track_id]['age'] < MAX_AGE:
            
            # Get this track's static data
            data = static_track_data[track_id]
            
            # Get the correct color (red for 1, yellow for 2, etc.)
            text_color = colors(track_id, True) 
            
            # Draw the lines
            cv2.putText(final_frame, f"track_ID: {track_id}", (start_x, current_y), 
                        font, font_scale, text_color, line_thickness, cv2.LINE_AA)
            current_y += line_height # Move down
            
            cv2.putText(final_frame, f"- lat: {data['lat']:.5f}", (start_x, current_y), 
                        font, font_scale, text_color, line_thickness, cv2.LINE_AA)
            current_y += line_height
            
            cv2.putText(final_frame, f"- lon: {data['lon']:.5f}", (start_x, current_y), 
                        font, font_scale, text_color, line_thickness, cv2.LINE_AA)
            current_y += line_height
            
            cv2.putText(final_frame, f"- alt: {data['alt']:.2f}", (start_x, current_y), 
                        font, font_scale, text_color, line_thickness, cv2.LINE_AA)
            current_y += line_height + 15 # Add extra spacing for the next track block


    # Display the resulting frame (every frame)
    cv2.imshow("Custom YOLOv8 Tracker (Fast)", final_frame)
    # out.write(final_frame) # (Optional)

    # Break the loop if 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# Release everything when job is finished
cap.release()
# out.release()
cv2.destroyAllWindows()
torch.cuda.empty_cache()
gc.collect()

print("Processing finished.")